# Problem Solution Pipeline 

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings("ignore")
 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, mean_absolute_error,
                             r2_score)

## 1. Data Preparation 
 DuckDB runs in-process — no server needed. We register each parquet file as a view so it can be queried with SQL directly without loading into memory.

In [ ]:
con = duckdb.connect()
 
con.execute("CREATE VIEW incident              AS SELECT * FROM read_parquet('../parquet-data/incident.parquet')")
con.execute("CREATE VIEW environmental         AS SELECT * FROM read_parquet('../parquet-data/environmental_conditions.parquet')")
con.execute("CREATE VIEW resource              AS SELECT * FROM read_parquet('../parquet-data/resource.parquet')")
con.execute("CREATE VIEW dispatch              AS SELECT * FROM read_parquet('../parquet-data/dispatch.parquet')")
 
print("✓ Views registered in DuckDB")
print(con.execute("SHOW TABLES").df())

## 2. Query Demonstration
Each query prepares a different analytical slice needed for the model or viz

In [ ]:
# Q1: Join all four tables into a single analytical flat file
df = con.execute("""
    SELECT
        i.incident_id,
        i.Timestamp,
        i.Incident_Type,
        i.Incident_Severity,
        i.Emergency_Level,
        i.Region_Type,
        i.Road_Type,
        i.Number_of_Injuries,
        i.Distance_to_Incident,
        i.Response_Time,
        i.Label,
        e.Weather_Condition,
        e.Weather_Impact,
        e.Traffic_Congestion,
        e.Air_Traffic,
        r.Drone_Availability,
        r.Ambulance_Availability,
        r.Battery_Life,
        r.Drone_Speed,
        r.Ambulance_Speed,
        r.Payload_Weight,
        r.Fuel_Level,
        d.Dispatch_Coordinator,
        d.Specialist_Availability,
        d.Hospital_Capacity
    FROM incident i
    JOIN environmental e ON i.incident_id = e.incident_id
    JOIN resource      r ON i.incident_id = r.incident_id
    JOIN dispatch      d ON i.incident_id = d.incident_id
""").df()
 
print(f"\n✓ Full join: {len(df):,} rows × {df.shape[1]} columns")

In [ ]:
# Q2: Average response time by region and dispatch label
q2 = con.execute("""
    SELECT
        i.Region_Type,
        i.Label,
        ROUND(AVG(i.Response_Time), 2) AS avg_response_min,
        COUNT(*)                        AS n_incidents
    FROM incident i
    GROUP BY i.Region_Type, i.Label
    ORDER BY avg_response_min
""").df()
print("\nQ2 — Avg response time by region & label:\n", q2)

In [ ]:
# Q3: Dispatch label distribution
q3 = con.execute("""
    SELECT Label, COUNT(*) AS n, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM incident
    GROUP BY Label
    ORDER BY n DESC
""").df()
print("\nQ3 — Label distribution:\n", q3)

In [ ]:
# Q4: Resource availability vs response time
q4 = con.execute("""
    SELECT
        r.Drone_Availability,
        r.Ambulance_Availability,
        ROUND(AVG(i.Response_Time), 2) AS avg_response_min,
        COUNT(*) AS n
    FROM incident i
    JOIN resource r ON i.incident_id = r.incident_id
    GROUP BY r.Drone_Availability, r.Ambulance_Availability
    ORDER BY avg_response_min
""").df()
print("\nQ4 — Availability vs response time:\n", q4)

In [ ]:
# Q5: Weather impact on response time
q5 = con.execute("""
    SELECT
        e.Weather_Impact,
        ROUND(AVG(i.Response_Time), 2) AS avg_response_min,
        COUNT(*) AS n
    FROM incident i
    JOIN environmental e ON i.incident_id = e.incident_id
    GROUP BY e.Weather_Impact
    ORDER BY avg_response_min DESC
""").df()
print("\nQ5 — Weather impact vs response time:\n", q5)

## 3. Feature Engineering
Extract time features from Timestamp — hour of day and day of week capture temporal patterns in incident volume and resource availability.

In [ ]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["hour"]      = df["Timestamp"].dt.hour
df["weekday"]   = df["Timestamp"].dt.weekday
 
CAT_FEATURES = [
    "Incident_Type", "Incident_Severity", "Emergency_Level",
    "Region_Type", "Road_Type",
    "Weather_Condition", "Weather_Impact", "Traffic_Congestion", "Air_Traffic",
    "Drone_Availability", "Ambulance_Availability",
    "Specialist_Availability", "Dispatch_Coordinator",
]
NUM_FEATURES = [
    "Number_of_Injuries", "Distance_to_Incident", "Battery_Life",
    "Drone_Speed", "Ambulance_Speed", "Payload_Weight",
    "Fuel_Level", "Hospital_Capacity", "hour", "weekday",
]
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES
 
X = df[ALL_FEATURES]
y_cls = df["Label"]
y_reg = df["Response_Time"]

# 4. Solution Analysis
Two models because the problem has two outputs: WHAT to dispatch (classification) and HOW LONG it will take (regression).
GradientBoosting handles mixed feature types and nonlinear interactions without requiring extensive preprocessing — important here because features like Traffic_Congestion interact with Drone_Speed in ways a linear model would miss.
StratifiedKFold ensures each fold has balanced Label representation, preventing evaluation bias toward the majority class.
StandardScaler on numerics + OHE on categoricals via ColumnTransformer keeps the pipeline reproducible and prevents data leakage.

In [ ]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ("num", StandardScaler(), NUM_FEATURES),
])
 
X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_cls, y_reg, test_size=0.2, random_state=42, stratify=y_cls
)

In [ ]:
# ── Classifier ────────────────────────────────────────────────────────────────
# n_estimators=200 and learning_rate=0.05 chosen for stable convergence;
# max_depth=5 balances expressiveness vs overfitting on a synthetic dataset.
clf_pipeline = Pipeline([
    ("pre", preprocessor),
    ("clf", GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42
    )),
])
clf_pipeline.fit(X_train, yc_train)
yc_pred = clf_pipeline.predict(X_test)
 
cv_scores = cross_val_score(clf_pipeline, X_train, yc_train,
                            cv=StratifiedKFold(5), scoring="accuracy")
print(f"\n✓ Classifier CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(classification_report(yc_test, yc_pred))

In [ ]:
# ── Regressor ─────────────────────────────────────────────────────────────────
reg_pipeline = Pipeline([
    ("pre", preprocessor),
    ("reg", GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42
    )),
])
reg_pipeline.fit(X_train, yr_train)
yr_pred = reg_pipeline.predict(X_test)
 
mae  = mean_absolute_error(yr_test, yr_pred)
r2   = r2_score(yr_test, yr_pred)
print(f"\n✓ Regressor  MAE: {mae:.2f} min  |  R²: {r2:.3f}")

## 5. Feature Importance
Extract feature names post-OHE for interpretability

In [ ]:
ohe_names = (clf_pipeline.named_steps["pre"]
             .named_transformers_["cat"]
             .get_feature_names_out(CAT_FEATURES))
feat_names = np.concatenate([ohe_names, NUM_FEATURES])
 
clf_imp = pd.Series(
    clf_pipeline.named_steps["clf"].feature_importances_, index=feat_names
).nlargest(12)
 
reg_imp = pd.Series(
    reg_pipeline.named_steps["reg"].feature_importances_, index=feat_names
).nlargest(12)

# ── 6. VISUALIZE RESULTS ──────────────────────────────────────────────────────
Rationale:
Panel A — Confusion matrix: directly shows where the classifier succeeds/fails per class, more informative than accuracy alone.
anel B — Predicted vs actual response time: standard regression diagnostic; diagonal alignment = good fit.
Panel C — Feature importances (classifier): shows WHICH signals drive dispatch recommendations — useful for human dispatcher trust.
Panel D — Avg response time by region & label (from Q2): connects model output back to the original operational question.
Panel E — Label distribution (from Q3): shows class balance, contextualizes classifier performance.
Panel F — Weather impact on response time (from Q5): domain-relevant insight surfaced by the SQL queries.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("IERAD — Model Results & Analysis", fontsize=15, fontweight="bold", y=1.01)
plt.subplots_adjust(hspace=0.45, wspace=0.35)
 
COLORS = ["#0F4C75", "#3282B8", "#BBE1FA", "#1B262C"]
 
# Panel A — Confusion matrix
ax = axes[0, 0]
cm = confusion_matrix(yc_test, yc_pred, labels=clf_pipeline.classes_)
disp = ConfusionMatrixDisplay(cm, display_labels=clf_pipeline.classes_)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("A — Dispatch Classifier\nConfusion Matrix", fontsize=11)
ax.tick_params(axis="x", labelrotation=20)
 
# Panel B — Predicted vs actual response time
ax = axes[0, 1]
ax.scatter(yr_test, yr_pred, alpha=0.3, s=12, color=COLORS[1])
lims = [min(yr_test.min(), yr_pred.min()), max(yr_test.max(), yr_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect fit")
ax.set_xlabel("Actual Response Time (min)")
ax.set_ylabel("Predicted Response Time (min)")
ax.set_title(f"B — Response Time Regressor\nMAE={mae:.2f} min  R²={r2:.3f}", fontsize=11)
ax.legend(fontsize=9)
 
# Panel C — Top 12 classifier feature importances
ax = axes[0, 2]
clf_imp.sort_values().plot(kind="barh", ax=ax, color=COLORS[0])
ax.set_title("C — Top Features\n(Dispatch Classifier)", fontsize=11)
ax.set_xlabel("Importance")
ax.tick_params(labelsize=8)
 
# Panel D — Avg response time by region & label
ax = axes[1, 0]
pivot = q2.pivot(index="Region_Type", columns="Label", values="avg_response_min")
pivot.plot(kind="bar", ax=ax, color=COLORS[:len(pivot.columns)], width=0.7)
ax.set_title("D — Avg Response Time\nby Region & Dispatch Type", fontsize=11)
ax.set_xlabel("")
ax.set_ylabel("Avg Response Time (min)")
ax.tick_params(axis="x", labelrotation=15)
ax.legend(fontsize=8, title="Label", title_fontsize=8)
 
# Panel E — Label distribution
ax = axes[1, 1]
bars = ax.bar(q3["Label"], q3["pct"], color=COLORS[:len(q3)])
ax.set_title("E — Dispatch Label Distribution", fontsize=11)
ax.set_ylabel("% of Incidents")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.tick_params(axis="x", labelrotation=15)
for bar, pct in zip(bars, q3["pct"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5, f"{pct}%",
            ha="center", va="bottom", fontsize=9)
 
# Panel F — Weather impact on response time
ax = axes[1, 2]
ax.bar(q5["Weather_Impact"], q5["avg_response_min"],
       color=[COLORS[0], COLORS[1], COLORS[2]])
ax.set_title("F — Weather Impact\nvs Avg Response Time", fontsize=11)
ax.set_ylabel("Avg Response Time (min)")
ax.set_xlabel("Weather Impact")
 
plt.tight_layout()
plt.savefig("ierad_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✓ Figure saved to ierad_results.png")
 